# 행렬 곱셈 순전파와 역전파

출처: [Triton 행렬 곱셈 튜토리얼](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html#sphx-glr-getting-started-tutorials-03-matrix-multiplication-py)

지난번에 구현한 행렬 곱셈을 다음과 같이 개선할 수 있습니다.

- 성능 향상을 위해 행 블록을 그룹화합니다.
- 역전파를 추가합니다.
- 활성화 함수를 커널에 융합합니다.

## 행 그룹화

여러 행 블록을 하나로 묶으면 동일한 열 블록을 여러 행 블록에서 재사용할 수 있습니다. 행 블록마다 열 블록을 반복해서 불러오는 대신 한 번만 불러오게 됩니다.

아래 그림을 참고하세요.

![블록 순서](images/grouped_vs_row_major_ordering.png)

메모리 접근이 병목이므로 이 방식으로 성능을 높일 수 있습니다.

## 활성화 함수 융합

행렬 곱셈 커널에서 활성화 함수를 호출해 연산을 융합할 수 있습니다. 먼저 활성화 함수를 정의합니다.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.testing import assert_close

import triton
import triton.language as tl

@triton.jit
def relu(x, shape):
    # x가 0보다 클 때만 x 값을 반환
    return tl.where(x > 0, x, 0.0)

이제 행렬 곱셈 커널의 실행 함수를 정의합니다. x와 y 프로그램 ID의 순서를 더 잘 제어할 수 있도록 1차원 그리드로 바꿉니다.

In [ ]:
def matmul_fwd(X, Y, activation="none"):
    x_rows, x_cols = X.shape
    y_rows, y_cols = Y.shape
    output = torch.zeros(x_rows, y_cols, device="cuda", dtype=torch.float16) # 출력 행렬
    
    BLOCK_SIZE_X = 128
    BLOCK_SIZE_Y = 128
    BLOCK_SIZE_K = 32
    GROUP_SIZE_X = 8
    
    # 블록을 순회하는 1차원 그리드 생성
    # 행 그룹화를 효율적으로 하기 위해 1차원 그리드를 사용
    grid = lambda meta: (triton.cdiv(x_rows, meta["BLOCK_SIZE_X"]) * triton.cdiv(y_cols, meta["BLOCK_SIZE_Y"]), )
    
    matmul_fwd_kernel[grid](X, Y, output, x_rows, x_cols, y_rows, y_cols, BLOCK_SIZE_X=BLOCK_SIZE_X, BLOCK_SIZE_Y=BLOCK_SIZE_Y, BLOCK_SIZE_K=BLOCK_SIZE_K, GROUP_SIZE_X=GROUP_SIZE_X, ACTIVATION=activation)
    
    return output

In [ ]:
@triton.jit
def matmul_fwd_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    x_rows,
    x_cols,
    y_rows,
    y_cols,
    BLOCK_SIZE_X: tl.constexpr, # 블록당 행 수
    BLOCK_SIZE_Y: tl.constexpr, # 블록당 열 수
    BLOCK_SIZE_K: tl.constexpr, # 반복마다 처리할 내적 차원의 크기
    GROUP_SIZE_X: tl.constexpr, # X 행 그룹의 크기
    ACTIVATION: tl.constexpr, # 사용할 활성화 함수
):
    # 전체 프로그램 ID 수는 X 블록 수와 Y 블록 수의 곱
    # 현재 프로그램의 ID를 가져옴
    pid = tl.program_id(axis=0)

    # X 방향의 블록 수
    num_pid_x = tl.cdiv(x_rows, BLOCK_SIZE_X)

    # Y 방향의 블록 수
    num_pid_y = tl.cdiv(y_cols, BLOCK_SIZE_Y)

    # 각 그룹에 포함된 프로그램 수
    # 행 그룹 수와 열 블록당 프로그램 수의 곱
    num_pid_in_group = GROUP_SIZE_X * num_pid_y

    # 현재 프로그램이 속한 그룹의 ID
    # 현재 pid를 그룹 내 프로그램 수로 나눠 그룹 ID를 구함
    group_id = pid // num_pid_in_group

    # 그룹에서 첫 번째 프로그램의 행 ID
    first_pid_x = group_id * GROUP_SIZE_X

    # 현재 그룹의 크기
    # num_pid_x가 GROUP_SIZE_X로 나누어떨어지지 않으면 마지막 그룹은 더 작음
    group_size_x = min(num_pid_x - first_pid_x, GROUP_SIZE_X)

    # 가져올 블록의 x pid. Y 열로 이동하기 전에 X의 행을 블록 단위로 순회
    # 동일한 열 블록을 여러 행 블록과 곱함
    # 실행 그리드에서 이 프로그램의 행 ID
    x_pid = first_pid_x + (pid % group_size_x)

    # 현재 그룹 안에서의 pid
    in_group_pid = (pid % num_pid_in_group) 

    # X의 행을 모두 순회한 뒤에야 Y 블록을 다음 열로 이동
    # 실행 그리드에서 이 프로그램의 열 ID
    y_pid = in_group_pid // group_size_x
    
    # X와 Y 포인터의 시작 위치
    # 행을 이동할 때는 열의 스트라이드를, 열을 이동할 때는 행의 스트라이드를 고려
    x_row_start = x_pid * BLOCK_SIZE_X * x_cols # X에서 선택할 블록의 시작 위치
    y_col_start = y_pid * BLOCK_SIZE_Y # Y 블록의 시작 위치
    
    # 행과 열의 오프셋. 행 오프셋에는 행렬의 열 수를 곱해야 함
    x_offsets = x_row_start + tl.arange(0, BLOCK_SIZE_X) * x_cols # 각 행의 시작 인덱스 (x_cols를 곱함)
    y_offsets = y_col_start + tl.arange(0, BLOCK_SIZE_Y) # 각 열의 시작 인덱스 (스트라이드는 1)
    
    # 행렬의 내적 차원에 대한 k 오프셋
    k_offsets = tl.arange(0, BLOCK_SIZE_K)
    
    # 각 행에서 0부터 k까지의 열을 가리키는 X 포인터
    x_ptrs = x_ptr + (x_offsets[:,None] + k_offsets[None,:])
    
    # 각 열에서 0부터 k까지의 행을 가리키는 Y 포인터
    # 행 시작 위치를 계산하기 위해 k 오프셋에 y_cols를 곱함
    y_ptrs = y_ptr + (k_offsets[:,None] * y_cols + y_offsets[None,:])
    
    # k를 따라 반복하며 결과를 누적
    # 수치 정밀도를 높이기 위해 float32로 저장
    accumulator = tl.zeros((BLOCK_SIZE_X, BLOCK_SIZE_Y), dtype=tl.float32)
    # BLOCK_SIZE_K씩 이동하며 k 차원 순회
    for k in range(0, tl.cdiv(x_cols, BLOCK_SIZE_K)):
        # 곱할 X 부분 블록 불러오기
        # 각 열의 끝을 넘는 데이터를 불러오지 않도록 마스크 적용
        # [None, :]은 첫 번째 차원에 길이 1을 추가해 마스크가 x_ptrs에 브로드캐스트되도록 함
        a = tl.load(x_ptrs, mask=k_offsets[None, :] < x_cols - k * BLOCK_SIZE_K, other=0.0)
        
        # [:, None]은 두 번째 차원에 길이 1을 추가해 마스크가 y_ptrs에 브로드캐스트되도록 함
        b = tl.load(y_ptrs, mask=k_offsets[:,None] < y_rows - k * BLOCK_SIZE_K, other=0.0)
        
        # a와 b를 곱한 뒤 누산기에 더함
        result = tl.dot(a, b)
        accumulator += result
        
        # 행을 따라 이동하도록 X 포인터 증가
        x_ptrs += BLOCK_SIZE_K
        
        # 열을 따라 이동하도록 Y 포인터 증가. 행 방향으로 이동하므로 y_cols를 곱함
        y_ptrs += BLOCK_SIZE_K * y_cols
    
    output = accumulator.to(tl.float16)
    
    # 활성화 함수 적용
    # 함수에 따라 fp16 변환 전후 중 한 시점에 적용 가능
    if ACTIVATION == "relu":
        output = relu(output, (BLOCK_SIZE_X, BLOCK_SIZE_Y))
    
    # 출력 포인터 위치 계산
    output_x_start = x_pid * BLOCK_SIZE_X
    output_y_start = y_pid * BLOCK_SIZE_Y
    
    output_x_rows = output_x_start + tl.arange(0, BLOCK_SIZE_X)
    
    output_x_offsets = output_x_start + tl.arange(0, BLOCK_SIZE_X)
    output_y_offsets = output_y_start + tl.arange(0, BLOCK_SIZE_Y)
    
    # 행렬 범위를 벗어나지 않도록 출력 데이터 저장
    output_ptrs = output_ptr + (output_x_offsets[:, None] * y_cols + output_y_offsets[None, :])
    output_mask = (output_x_rows[:, None] < x_rows) & (output_y_offsets[None,:] < y_cols)
    tl.store(output_ptrs, output, mask=output_mask)

In [ ]:
torch.manual_seed(0)
x = torch.rand((8192, 8192), device='cuda', dtype=torch.float16) - .5
y = torch.rand((8192, 8192), device='cuda', dtype=torch.float16) - .5

act = nn.ReLU()
output_torch = act(x @ y)
output_triton = matmul_fwd(x, y, activation="relu")
print(
    f'PyTorch와 Triton 결과의 최댓값 차이: '
    f'{torch.max(torch.abs(output_torch - output_triton))}'
)
print(output_torch[:5,:5])
print(output_triton[:5,:5])

The maximum difference between torch and triton is 0.0
tensor([[ 0.0000,  0.0000,  0.0000,  5.2227,  0.0000],
        [ 2.6484,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 5.5938,  0.0000,  0.0000,  2.3438,  9.6562],
        [ 0.0000,  3.9062,  0.0000,  0.0000,  0.0000],
        [14.8203,  8.7969,  0.0000,  0.2194,  0.0000]], device='cuda:0',
       dtype=torch.float16)
tensor([[ 0.0000,  0.0000,  0.0000,  5.2227,  0.0000],
        [ 2.6484,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 5.5938,  0.0000,  0.0000,  2.3438,  9.6562],
        [ 0.0000,  3.9062,  0.0000,  0.0000,  0.0000],
        [14.8203,  8.7969,  0.0000,  0.2194,  0.0000]], device='cuda:0',
       dtype=torch.float16)


## 역전파 추가

`Z = X @ Y`일 때 `X`에 대한 그래디언트는 `Z @ Y.T`, `Y`에 대한 그래디언트는 `X.T @ Z`입니다. 행렬 곱셈 커널을 두 번 호출해 계산할 수 있습니다. 두 연산 모두에서 `Z`를 그룹당 한 번만 불러오도록 추가 최적화할 수도 있습니다.

먼저 성능 비교를 위해 PyTorch에서 같은 연산을 수행합니다.

In [5]:
x = torch.rand((8192, 8192), device='cuda', dtype=torch.float16) - .5
y = torch.rand((8192, 8192), device='cuda', dtype=torch.float16) - .5

x.requires_grad = True
y.requires_grad = True

output_torch = act(x @ y)
output_torch = F.softmax(output_torch, dim=-1)

In [6]:
target = torch.zeros_like(x)
inds = (torch.arange(0, target.shape[0]), torch.randint(0, x.shape[1], (target.shape[0],)))
target[inds] = 1

loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(output_torch, target)
loss.backward()

In [7]:
torch_xgrad = x.grad.detach().clone()
torch_ygrad = y.grad.detach().clone()

이제 행렬 곱셈 커널을 정의합니다.

In [ ]:
class MatMul(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, y):
        activation = "relu"
        z = matmul_fwd(x, y, activation)
        
        ctx.save_for_backward(x, y, z) # 역전파에서 사용할 x, y, z를 캐시에 저장
        ctx.activation = activation
        return z

    @staticmethod
    def backward(ctx, dz):
        x, y, z = ctx.saved_tensors # 캐시에서 x, y, z 가져오기
        
        # ReLU 역전파 적용
        # 이 연산은 커널에 융합하면 더 효율적
        if ctx.activation == "relu":
            dz = torch.where(z > 0, dz, 0.0)
        
        # 여기서는 행렬 곱셈을 한 번만 호출하도록 구현하면 더 효율적
        dx = matmul_fwd(dz, y.T, activation="none")
        dy = matmul_fwd(x.T, dz, activation="none")
        
        return dx, dy

matmul = MatMul.apply

In [ ]:
x.grad.data.zero_() # x의 그래디언트를 0으로 초기화
y.grad.data.zero_() # y의 그래디언트를 0으로 초기화

output_triton = matmul(x, y)
output_triton = F.softmax(output_triton, dim=-1)

triton_loss = loss_fn(output_triton, target)
triton_loss.backward()
assert_close(x.grad, torch_xgrad, atol=1e-2, rtol=0)
assert_close(y.grad, torch_ygrad, atol=1e-2, rtol=0)